# M3.S4 — Libraries, I/O & Reproducible Workflows

**HPC 2026 · Session 14 · Live notebook**  
**Build:** `M3S4-2026-09-27-v3`

This notebook is not a chapter to read from top to bottom. We will return to it at specific points in the slides.

> **PREDICT → RUN → OBSERVE → EXPLAIN**

Today we connect four parts of real HPC work:

1. **USE** mature, optimized libraries instead of rebuilding standard kernels.
2. **MOVE DATA** with an I/O design that matches the workload.
3. **RECORD** enough information to explain exactly what was run.
4. **REPEAT** the experiment under a comparable software and hardware environment.

**Important:** small timing experiments in this notebook are teaching experiments, not publication-quality benchmarks.

<details>
<summary><strong>Instructor route (click to expand)</strong></summary>

- **Slides 2–4:** use-vs-build and the HPC software stack.
- **Slides 5–6:** Experiment 1, handwritten DFT vs library FFT.
- **Module slide:** inspect/load software with environment modules and record the environment.
- **I/O slides:** I/O decision game + many-small-files experiment.
- **Storage / structured I/O:** inspect the current storage path + HDF5 / MPI-IO.
- **Reproducibility slides:** reproducibility failure + build a run manifest.
- **Portability slide:** native modules → container → workflow, concept only.
- **Recap / exit:** connect the manifest to Practice 3.

Suggested rhythm: let students **predict before every result is shown**. Do not treat library names or module commands as a catalogue to memorize.
</details>


## 0 — Pre-flight: what environment are we actually using?

Before timing anything, record the environment. This is already part of reproducibility.

In [ ]:
from pathlib import Path
import os, sys, platform, shutil, subprocess, time, statistics, hashlib, json, math, cmath, tempfile
from datetime import datetime, timezone

import numpy as np

BUILD = "M3S4-2026-09-27-v3"
# Use a user-owned workspace. Fall back to a temporary directory if needed.
preferred_work = Path.home() / "m3s4_workspace"
try:
    preferred_work.mkdir(parents=True, exist_ok=True)
    test_file = preferred_work / ".write_test"
    test_file.write_text("ok")
    test_file.unlink()
    WORK = preferred_work
except Exception:
    WORK = Path(tempfile.mkdtemp(prefix="m3s4_workspace_"))

print("Notebook build :", BUILD)
print("Python         :", sys.version.split()[0])
print("NumPy          :", np.__version__)
print("Host           :", platform.node())
print("Working dir    :", Path.cwd())
print("Workspace      :", WORK)
print("SLURM_JOB_ID   :", os.getenv("SLURM_JOB_ID", "not inside a Slurm job"))

### Retrieval question

Without looking back at the slides, classify each task:

| Task | First instinct |
|---|---|
| Multiply two large dense matrices | Library or handwritten? |
| Compute a Fourier transform | Library or handwritten? |
| Implement a genuinely new research algorithm | Library or handwritten? |

<details>
<summary><strong>Concise answer</strong></summary>

Standard kernels such as dense matrix operations and FFTs are normally delegated to mature libraries. A genuinely new algorithm may require custom code, but that code can still call optimized libraries internally.
</details>

# 1 — REAL EXPERIMENT: handwritten DFT vs library FFT

## The question

Suppose we need the frequency spectrum of a signal.

A direct discrete Fourier transform (DFT) evaluates every output frequency against every input sample: roughly **O(N²)** work.

A fast Fourier transform (FFT) uses a better algorithm: roughly **O(N log N)** work.

### PREDICT

Before running the next cells:

1. Which implementation should grow faster as `N` increases?
2. Will both produce approximately the same numerical result?
3. Is a library faster *only* because its code is better optimized?

In [ ]:
def naive_dft(x):
    """Clear O(N^2) teaching implementation of the DFT."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    out = np.empty(n, dtype=complex)
    for k in range(n):
        total = 0j
        for t, value in enumerate(x):
            total += value * cmath.exp(-2j * math.pi * k * t / n)
        out[k] = total
    return out

# Small correctness check first.
rng = np.random.default_rng(2026)
x = rng.normal(size=32)
manual = naive_dft(x)
library = np.fft.fft(x)

max_error = float(np.max(np.abs(manual - library)))
print(f"Maximum absolute difference: {max_error:.3e}")
print("Correct within tolerance    :", np.allclose(manual, library, rtol=1e-10, atol=1e-10))

### RUN

We now measure both methods for increasing input sizes.

For the very fast library call we repeat the timing several times and keep the median. The direct DFT is slow enough that one run per size is sufficient for this classroom comparison.

In [ ]:
def median_time(fn, *args, repeats=5):
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn(*args)
        samples.append(time.perf_counter() - t0)
    return statistics.median(samples), result

sizes = [64, 128, 256, 512, 1024]
fft_results = []

for n in sizes:
    x = np.random.default_rng(2026).normal(size=n)

    t0 = time.perf_counter()
    y_direct = naive_dft(x)
    direct_s = time.perf_counter() - t0

    fft_s, y_fft = median_time(np.fft.fft, x, repeats=9)
    err = float(np.max(np.abs(y_direct - y_fft)))

    fft_results.append((n, direct_s, fft_s, direct_s / fft_s, err))

print(f"{'N':>6} {'Direct DFT (s)':>16} {'FFT median (s)':>16} {'Ratio':>12} {'Max error':>12}")
for n, direct_s, fft_s, ratio, err in fft_results:
    print(f"{n:6d} {direct_s:16.6f} {fft_s:16.8f} {ratio:12.1f} {err:12.2e}")

In [ ]:
# Optional visual: how the measured times grow.
try:
    import matplotlib.pyplot as plt

    ns = [r[0] for r in fft_results]
    direct = [r[1] for r in fft_results]
    fft = [r[2] for r in fft_results]

    plt.figure(figsize=(7, 4))
    plt.plot(ns, direct, marker='o', label='Direct DFT')
    plt.plot(ns, fft, marker='o', label='NumPy FFT')
    plt.yscale('log')
    plt.xlabel('N')
    plt.ylabel('Measured wall time (s, log scale)')
    plt.title('Same mathematical result, different algorithm/implementation')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()
except Exception as exc:
    print("Plot skipped:", exc)

### OBSERVE → EXPLAIN

Write one sentence for each:

- **Correctness:** Did both implementations agree?
- **Growth:** Which curve grows more rapidly?
- **Why:** Which part of the difference is algorithmic and which part is implementation quality?

<details>
<summary><strong>Discussion checkpoint</strong></summary>

The direct DFT and FFT should agree numerically up to floating-point error. The direct implementation scales roughly quadratically, while the FFT has much better asymptotic complexity. NumPy also calls highly optimized compiled code, so both **algorithm choice** and **implementation quality** contribute to the measured gap.
</details>

# 2 — REAL CLUSTER CHECK: environment modules

Environment modules are a standard way to select compilers, MPI implementations and scientific libraries on HPC systems.

They **do not copy software into your home directory**. Instead, a module changes environment variables such as `PATH`, `LD_LIBRARY_PATH`, `CPATH` and library-specific settings.

Typical workflow:

```bash
module avail
module spider FFTW
module load FFTW/3.3.10-GCC-13.2.0
module list
```

At the end of an experiment, the loaded module set is part of the evidence we should record.

### PREDICT

If two students run the same source code with different compiler/library modules, must they obtain identical performance?


In [ ]:
def run_bash(script):
    p = subprocess.run(
        ["bash", "-lc", script],
        text=True,
        capture_output=True,
        timeout=30,
    )
    out = (p.stdout or "") + (p.stderr or "")
    return p.returncode, out.strip()

EESSI_INIT = "/cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash"

probe = rf"""
if command -v module >/dev/null 2>&1; then
    echo "MODULES_STATUS=already_available"
elif [ -r "{EESSI_INIT}" ]; then
    source "{EESSI_INIT}"
    echo "MODULES_STATUS=EESSI_initialized"
else
    echo "MODULES_STATUS=not_available_in_this_environment"
    exit 0
fi

echo
echo "--- module tool ---"
module --version 2>&1 | head -n 4

echo
echo "--- currently loaded modules ---"
module -t list 2>&1 || true

echo
echo "--- FFTW modules visible here ---"
module -t avail FFTW 2>&1 | head -n 20 || true
"""

rc, module_probe_output = run_bash(probe)
print(module_probe_output)


### OBSERVE

On SciTech, the notebook can initialize the **EESSI** software environment from CVMFS if the module command is not already active.

For EESSI 2023.06, FFTW 3.3.10 is available as an environment module. The exact available build matters because the module name also identifies a compiler/toolchain combination.

Now we load one real FFTW module **inside a temporary shell**. This does not alter the Python kernel that is running the notebook.


In [ ]:
load_demo = rf"""
if command -v module >/dev/null 2>&1; then
    :
elif [ -r "{EESSI_INIT}" ]; then
    source "{EESSI_INIT}"
else
    echo "MODULE_LOAD_DEMO=SKIPPED (module system not available here)"
    exit 0
fi

echo "--- before ---"
module -t list 2>&1 || true

MODULE_NAME="FFTW/3.3.10-GCC-13.2.0"
echo
echo "Loading: $MODULE_NAME"

if module load "$MODULE_NAME" >/dev/null 2>&1; then
    echo "MODULE_LOAD_STATUS=OK"
    echo
    echo "--- after ---"
    module -t list 2>&1 || true
    echo
    echo "fftw-wisdom path:"
    command -v fftw-wisdom || true
else
    echo "MODULE_LOAD_STATUS=SKIPPED"
    echo "That exact module build is not visible on this node."
    echo "Use: module avail FFTW"
fi
"""

rc, module_load_output = run_bash(load_demo)
print(module_load_output)


### EXPLAIN

A useful mental model is:

```text
module load
    ↓
select software/toolchain
    ↓
run experiment
    ↓
module list
    ↓
record environment in the run manifest
```

A Slurm job can record its module environment with:

```bash
module -t list 2> modules.txt
```

or, for machine-readable provenance, with the `LOADEDMODULES` environment variable when the module system provides it.

> **The module list is part of a reproducible performance experiment.**

We will return to modules, Apptainer and portability in much more depth in Session 23.


## 2A — Discover, inspect and control modules

The four commands students should know first are:

```bash
module avail          # what can I load?
module spider FFTW    # where is a package/version?
module show FFTW/...  # what will this module change?
module list           # what is active now?
```

Two more are useful when experimenting:

```bash
module unload NAME
module purge
```

### PREDICT

If `module load FFTW/...` changes `PATH`, `CPATH` and `LD_LIBRARY_PATH`, what should happen after `module unload`?

> The software should no longer be selected through that module environment.


In [ ]:
inspect_demo = rf"""
if command -v module >/dev/null 2>&1; then
    :
elif [ -r "{EESSI_INIT}" ]; then
    source "{EESSI_INIT}"
else
    echo "MODULE_INSPECT_DEMO=SKIPPED"
    exit 0
fi

echo "=== module spider FFTW ==="
module spider FFTW 2>&1 | head -n 35 || true

echo
echo "=== choose a visible FFTW module ==="
FFTW_MOD=$(module -t avail FFTW 2>&1 \
  | sed 's/\x1b\[[0-9;]*m//g' \
  | grep -E '^FFTW/[0-9]' \
  | head -n 1)

echo "Selected: ${{FFTW_MOD:-NONE}}"

if [ -n "$FFTW_MOD" ]; then
    echo
    echo "=== module show ==="
    module show "$FFTW_MOD" 2>&1 | head -n 45 || true
fi
"""

rc, module_inspect_output = run_bash(inspect_demo)
print(module_inspect_output)


### OBSERVE

`module show` is especially useful because it exposes what the module is doing to the environment.

Typical changes include:

- executable search paths;
- library search paths;
- include/header paths;
- dependent compiler/runtime modules.

This is why **the module name and version are part of the experiment configuration**, not just a convenience command.


## 2B — REAL EXAMPLE: compile and run code using a loaded FFTW module

Now we use the module for something real.

The notebook writes a tiny C program that:

1. creates an FFTW plan;
2. transforms eight real input values;
3. prints the first few complex output coefficients.

Without the FFTW development environment, the compiler may not find `fftw3.h` or the linker may not find `libfftw3`.

With the appropriate module loaded, the same shell can compile and run the program.


In [ ]:
fftw_source = WORK / "m3s4_fftw_demo.c"
fftw_source.write_text(r"""
#include <stdio.h>
#include <fftw3.h>

int main(void) {
    const int N = 8;
    double in[N];
    fftw_complex out[N/2 + 1];

    for (int i = 0; i < N; ++i)
        in[i] = (double)(i + 1);

    fftw_plan p = fftw_plan_dft_r2c_1d(N, in, out, FFTW_ESTIMATE);
    if (!p) {
        fprintf(stderr, "Could not create FFTW plan\n");
        return 2;
    }

    fftw_execute(p);

    printf("FFTW_DEMO N=%d\n", N);
    for (int k = 0; k < N/2 + 1; ++k)
        printf("k=%d  real=% .6f  imag=% .6f\n",
               k, out[k][0], out[k][1]);

    fftw_destroy_plan(p);
    return 0;
}
""")

compile_run = rf"""
if command -v module >/dev/null 2>&1; then
    :
elif [ -r "{EESSI_INIT}" ]; then
    source "{EESSI_INIT}"
else
    echo "FFTW_REAL_DEMO=SKIPPED (module system unavailable)"
    exit 0
fi

FFTW_MOD=$(module -t avail FFTW 2>&1 \
  | sed 's/\x1b\[[0-9;]*m//g' \
  | grep -E '^FFTW/[0-9]' \
  | head -n 1)

if [ -z "$FFTW_MOD" ]; then
    echo "FFTW_REAL_DEMO=SKIPPED (no FFTW module visible)"
    exit 0
fi

echo "Loading: $FFTW_MOD"
module load "$FFTW_MOD"

echo
echo "Loaded modules:"
module -t list 2>&1 || true

echo
echo "Compiler:"
command -v gcc || true
gcc --version | head -n 1 || true

echo
echo "Compile:"
gcc "{fftw_source}" -lfftw3 -lm -o "{WORK / 'm3s4_fftw_demo'}"
rc=$?
echo "COMPILE_RC=$rc"

if [ "$rc" -eq 0 ]; then
    echo
    echo "Run:"
    "{WORK / 'm3s4_fftw_demo'}"
fi
"""

rc, fftw_real_output = run_bash(compile_run)
print(fftw_real_output)


### EXPLAIN

This is the complete HPC-software workflow students should recognize:

```text
discover software
      ↓
module load FFTW/...
      ↓
compile against FFTW
      ↓
run the program
      ↓
record module list + compiler + command
```

The module did **not** replace the source code or algorithm.

It supplied a controlled software environment in which the compiler could find the library.


## 2C — Modules inside a Slurm job

For real performance work, record the environment **inside the job** that performed the measurement.

A minimal batch script pattern is:

```bash
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --cpus-per-task=4

module purge
module load GCC/...
module load FFTW/...

module -t list 2> modules.txt
gcc --version > compiler.txt

srun ./my_program
```

Why inside the job?

> The environment that matters is the environment in which the measured program actually ran.

### Mini-task

Which files would you keep with the result?

- `modules.txt`
- `compiler.txt`
- Slurm script
- program output/timing
- code commit / manifest

**Answer: all of them.**


# 3 — DECIDE: four ways to write output

A simulation runs with **64 ranks** and periodically writes results.

### PREDICT

For each design, identify one advantage and one risk.

**A — Gather to rank 0**  
64 ranks → rank 0 → one file

**B — File per rank**  
64 ranks → 64 files

**C — Everyone appends to the same text file**  
64 ranks → same file

**D — Collective structured I/O**  
64 ranks → HDF5 / MPI-IO → shared dataset

Do not look for a universal “best” answer. The point is to reason about **serialization, metadata, contention, coordination and structure**.

<details>
<summary><strong>Discussion checkpoint</strong></summary>

- **A:** simple and often safe, but rank 0 can become a serialization and memory bottleneck.
- **B:** easy to parallelize, but can create large file counts and metadata pressure.
- **C:** can create contention and ordering/correctness problems unless carefully coordinated.
- **D:** can scale well for structured scientific output, but the result still depends on library configuration, access pattern and filesystem behavior.

The correct design depends on the workload and system.
</details>

# 4 — REAL EXPERIMENT: why many small files can be expensive

The total number of bytes is not the only cost of I/O.

Every file normally implies operations such as **create/open → metadata → write → close**.

We will compare:

- many small files carrying the same total payload;
- one larger file carrying that payload.

### Safety

This experiment uses a temporary local directory and only a modest number of files. **Do not turn it into a “10,000 files” experiment on a shared production filesystem.**

In [ ]:
FILES_TO_CREATE = 300
BYTES_PER_FILE = 4096
PAYLOAD = b'x' * BYTES_PER_FILE
TOTAL_BYTES = FILES_TO_CREATE * BYTES_PER_FILE

print("Files in many-file case:", FILES_TO_CREATE)
print("Bytes per small file     :", BYTES_PER_FILE)
print("Total payload            :", TOTAL_BYTES, "bytes")
print("One-file payload         :", TOTAL_BYTES, "bytes")

### PREDICT

If the total payload is the same, will the two cases take exactly the same time? Which case performs more file-management operations?

In [ ]:
def write_many_small_files(root, n_files=FILES_TO_CREATE, payload=PAYLOAD):
    for i in range(n_files):
        (root / f"part_{i:04d}.bin").write_bytes(payload)


def write_one_file(root, n_files=FILES_TO_CREATE, payload=PAYLOAD):
    (root / "all_data.bin").write_bytes(payload * n_files)


def time_io_case(fn, repeats=5):
    samples = []
    for _ in range(repeats):
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp)
            t0 = time.perf_counter()
            fn(root)
            samples.append(time.perf_counter() - t0)
    return statistics.median(samples), samples

small_median, small_samples = time_io_case(write_many_small_files)
one_median, one_samples = time_io_case(write_one_file)

print(f"Many small files, median : {small_median:.6f} s")
print(f"One larger file, median  : {one_median:.6f} s")
print(f"Observed ratio           : {small_median / one_median:.1f}x")
print()
print("Interpretation: this is a LOCAL teaching experiment, not a parallel-filesystem benchmark.")

### Optional: inspect system calls with `strace`

If `strace` is available in the environment, the next cell runs two tiny writers under `strace -c` and prints the syscall summaries. This is the **I/O trace** for the session.

The important question is not the exact runtime. Look at the number of calls associated with opening/creating/writing/closing files.

In [ ]:
strace = shutil.which("strace")
python_exe = sys.executable

small_script = WORK / "trace_many_small.py"
one_script = WORK / "trace_one_file.py"

small_script.write_text('''from pathlib import Path\nimport tempfile\npayload=b"x"*4096\nwith tempfile.TemporaryDirectory() as d:\n    p=Path(d)\n    for i in range(50):\n        (p/f"f_{i}.bin").write_bytes(payload)\n''')

one_script.write_text('''from pathlib import Path\nimport tempfile\npayload=b"x"*(50*4096)\nwith tempfile.TemporaryDirectory() as d:\n    p=Path(d)\n    (p/"one.bin").write_bytes(payload)\n''')

if not strace:
    print("strace is not available here. Use the timing experiment above or a pre-recorded trace in class.")
else:
    for label, script in [("MANY SMALL FILES", small_script), ("ONE FILE", one_script)]:
        print("\n" + "="*70)
        print(label)
        print("="*70)
        proc = subprocess.run(
            [strace, "-c", python_exe, str(script)],
            text=True,
            capture_output=True,
            timeout=30,
        )
        print(proc.stderr)

### OBSERVE → EXPLAIN

Why can a “small-file storm” hurt an HPC system even when the total data volume is modest?

<details>
<summary><strong>Concise answer</strong></summary>

Because many files create many namespace and metadata operations in addition to the actual data writes. On a shared filesystem, many ranks doing this concurrently can stress metadata services and create contention even before bulk storage bandwidth is saturated.
</details>

# 5 — REAL SYSTEM CHECK: where are we writing?

I/O numbers are meaningless if we do not know what storage path we measured.

The next cell reports the host and asks the operating system about the filesystem containing the current directory.

In [ ]:
print("Host:", platform.node())
print("Current directory:", Path.cwd())

if shutil.which("df"):
    proc = subprocess.run(["df", "-T", "."], text=True, capture_output=True)
    print("\nFilesystem for current directory:")
    print(proc.stdout)
else:
    usage = shutil.disk_usage(Path.cwd())
    print("Disk usage:", usage)

print("Important: interpret the output using the actual cluster documentation. Do not infer 'parallel filesystem' from a path name alone.")

### DISCUSS

If this notebook is running on the university HPC environment:

1. Is this directory local to the notebook server, shared across nodes, or backed by another storage service?
2. Would you expect the same timing in `/tmp`, your home directory and a parallel scratch filesystem?
3. What would you need to record before reporting an I/O benchmark?

# 6 — REAL EXAMPLE: structured scientific data with HDF5

Scientific output is often more than a byte stream. We may want named arrays, dimensions, units and other metadata.

We will create one small HDF5 file containing a time series of 2-D temperature fields.

In [ ]:
try:
    import h5py
    H5PY_AVAILABLE = True
    print("h5py version:", h5py.__version__)
except Exception as exc:
    H5PY_AVAILABLE = False
    print("h5py is not available:", exc)

In [ ]:
h5_path = WORK / "simulation.h5"

if H5PY_AVAILABLE:
    rng = np.random.default_rng(2026)
    temperature = 273.15 + 15.0 * rng.random((4, 64, 64))

    with h5py.File(h5_path, "w") as f:
        f.attrs["experiment"] = "M3S4 teaching example"
        f.attrs["created_by"] = BUILD

        fields = f.create_group("fields")
        dset = fields.create_dataset(
            "temperature",
            data=temperature,
            chunks=(1, 64, 64),
            compression="gzip",
            compression_opts=1,
        )
        dset.attrs["units"] = "K"
        dset.attrs["dimensions"] = "time,y,x"
        dset.attrs["dt_seconds"] = 60.0

    print("Created:", h5_path)
    print("File size:", h5_path.stat().st_size, "bytes")
else:
    print("Skipping HDF5 creation.")

In [ ]:
if H5PY_AVAILABLE:
    with h5py.File(h5_path, "r") as f:
        print("File attributes:")
        for k, v in f.attrs.items():
            print(f"  {k}: {v}")

        print("\nHierarchy:")
        def show(name, obj):
            kind = "GROUP" if isinstance(obj, h5py.Group) else "DATASET"
            print(f"  {kind:7s} /{name}")
        f.visititems(show)

        ds = f["/fields/temperature"]
        print("\nDataset shape:", ds.shape)
        print("Dataset units:", ds.attrs["units"])
        print("First value  :", float(ds[0, 0, 0]))

### Key idea

HDF5 gives us a **structured data model**. MPI-IO is a lower-level parallel file-access interface. Parallel HDF5 implementations can use MPI-IO underneath.

Think in layers:

> **APPLICATION → HDF5 / NetCDF → MPI-IO → FILESYSTEM → STORAGE**

The APIs are not competing answers to the same question; they operate at different abstraction levels.

# 7 — OPTIONAL REAL CLUSTER DEMO: collective MPI-IO

This section writes a tiny **real MPI program**. Each MPI rank writes one block into a different region of a single shared file using `MPI_File_write_at_all`.

It is intentionally optional because the exact MPI module/toolchain varies between systems.

**Safe defaults:** one node, 4 MPI ranks, 4 KiB total output, short wall time.

In [ ]:
mpi_c = r'''#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    const int block = 1024;
    char *buf = malloc(block);
    memset(buf, 'A' + (rank % 26), block);

    MPI_File fh;
    MPI_File_open(MPI_COMM_WORLD, "mpi_collective_output.bin",
                  MPI_MODE_CREATE | MPI_MODE_WRONLY,
                  MPI_INFO_NULL, &fh);

    MPI_File_set_size(fh, 0);
    MPI_Barrier(MPI_COMM_WORLD);

    MPI_Offset offset = (MPI_Offset)rank * block;
    MPI_File_write_at_all(fh, offset, buf, block, MPI_CHAR, MPI_STATUS_IGNORE);
    MPI_File_close(&fh);

    if (rank == 0) {
        printf("Wrote %d ranks x %d bytes = %d bytes to one shared file\n",
               size, block, size * block);
    }

    free(buf);
    MPI_Finalize();
    return 0;
}
'''

sbatch_text = r'''#!/bin/bash
#SBATCH --job-name=m3s4_mpiio
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --time=00:03:00
#SBATCH --output=m3s4_mpiio-%j.out

set -euo pipefail
srun -n 4 ./mpi_io_demo
ls -lh mpi_collective_output.bin
'''

mpi_source = WORK / "mpi_io_demo.c"
mpi_job = WORK / "m3s4_mpiio.sbatch"
mpi_source.write_text(mpi_c)
mpi_job.write_text(sbatch_text)

print("Created:", mpi_source)
print("Created:", mpi_job)
print("mpicc:", shutil.which("mpicc") or "not detected")
print("sbatch:", shutil.which("sbatch") or "not detected")
print("srun  :", shutil.which("srun") or "not detected")

### Optional submission

Set `RUN_CLUSTER_DEMO = True` **only when the instructor wants to submit the tiny job**.

If `mpicc` is not on the path, load the MPI module appropriate for the university cluster first. This notebook deliberately does not guess the module name.

In [ ]:
RUN_CLUSTER_DEMO = False

if not RUN_CLUSTER_DEMO:
    print("Cluster submission disabled. Set RUN_CLUSTER_DEMO = True when ready.")
else:
    mpicc = shutil.which("mpicc")
    sbatch = shutil.which("sbatch")
    srun = shutil.which("srun")

    if not (mpicc and sbatch and srun):
        print("Required tools are not all available. Need mpicc, sbatch and srun.")
    else:
        compile_proc = subprocess.run(
            [mpicc, "-O2", "-o", "mpi_io_demo", "mpi_io_demo.c"],
            cwd=WORK,
            text=True,
            capture_output=True,
        )
        if compile_proc.returncode != 0:
            print("Compilation failed:")
            print(compile_proc.stderr)
        else:
            submit = subprocess.run(
                [sbatch, "m3s4_mpiio.sbatch"],
                cwd=WORK,
                text=True,
                capture_output=True,
            )
            print(submit.stdout.strip() or submit.stderr.strip())
            if submit.returncode == 0:
                job_id = submit.stdout.strip().split()[-1]
                print("Submitted job:", job_id)
                print("Use squeue -j", job_id, "to follow it.")

# 8 — REPRODUCIBILITY FAILURE: code is not enough

Imagine that last month someone reported:

```text
Runtime: 12.8 s
Result : 0.7421
```

They kept the source file, but not the exact run information.

### PREDICT

What could prevent us from reproducing either the **scientific result** or the **performance result**?

In [ ]:
# Create a deterministic input dataset that we can identify by checksum.
input_path = WORK / "input.npy"
input_data = np.random.default_rng(12345).normal(size=250_000)
np.save(input_path, input_data)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Input file:", input_path)
print("SHA-256  :", sha256_file(input_path))

In [ ]:
def poorly_recorded_run(data):
    # Deliberately bad for reproducibility: seed is not recorded.
    rng = np.random.default_rng()
    noise = rng.normal(0.0, 0.01, size=data.shape)
    return float(np.mean((data + noise) ** 2))

r1 = poorly_recorded_run(input_data)
r2 = poorly_recorded_run(input_data)

print("Run 1:", r1)
print("Run 2:", r2)
print("Exactly identical?", r1 == r2)

The code did not change, but the result did.

A missing random seed is only one example. A performance result can also change because of compiler/library versions, thread/rank counts, node choice, CPU/GPU model, environment variables, input version, scheduler allocation or competing system load.

> **A reproducible run is an evidence package, not just a source file.**

# 9 — BUILD: a minimum run manifest

For this course, a minimum manifest should record at least:

- **run ID / timestamp**
- **code identity** — Git commit when available
- **input identity** — filename + checksum/version
- **command and parameters**
- **random seed** when relevant
- **software environment** — Python/compiler/libraries/modules
- **resources** — nodes/ranks/threads/GPUs
- **outputs and timing**

We now create a machine-readable JSON manifest.

In [ ]:
def git_commit_or_none(path=Path.cwd()):
    try:
        p = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=path,
            text=True,
            capture_output=True,
            timeout=3,
        )
        return p.stdout.strip() if p.returncode == 0 else None
    except Exception:
        return None


def reproducible_simulation(data, seed, noise_sigma=0.01):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, noise_sigma, size=data.shape)
    return float(np.mean((data + noise) ** 2))

seed = 2026
noise_sigma = 0.01

t0 = time.perf_counter()
result = reproducible_simulation(input_data, seed=seed, noise_sigma=noise_sigma)
elapsed = time.perf_counter() - t0

manifest = {
    "run_id": datetime.now(timezone.utc).strftime("M3S4-%Y%m%dT%H%M%SZ"),
    "notebook_build": BUILD,
    "code": {
        "git_commit": git_commit_or_none(),
        "entry_point": "reproducible_simulation() in this notebook",
    },
    "input": {
        "path": str(input_path),
        "sha256": sha256_file(input_path),
        "elements": int(input_data.size),
    },
    "parameters": {
        "seed": seed,
        "noise_sigma": noise_sigma,
    },
    "software": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "h5py": getattr(globals().get("h5py", None), "__version__", None),
        "loaded_modules": [m for m in os.getenv("LOADEDMODULES", "").split(":") if m],
        "modulepath": os.getenv("MODULEPATH"),
    },
    "system": {
        "hostname": platform.node(),
        "platform": platform.platform(),
    },
    "resources": {
        "slurm_job_id": os.getenv("SLURM_JOB_ID"),
        "slurm_nodes": os.getenv("SLURM_JOB_NUM_NODES"),
        "slurm_ntasks": os.getenv("SLURM_NTASKS"),
        "omp_num_threads": os.getenv("OMP_NUM_THREADS"),
        "cuda_visible_devices": os.getenv("CUDA_VISIBLE_DEVICES"),
    },
    "result": {
        "value": result,
        "elapsed_seconds": elapsed,
    },
}

manifest_path = WORK / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print(manifest_path.read_text())

### RERUN from recorded evidence

A manifest is useful only if it lets us reconstruct the run.

In [ ]:
loaded = json.loads(manifest_path.read_text())
reloaded_input = np.load(loaded["input"]["path"])

assert sha256_file(loaded["input"]["path"]) == loaded["input"]["sha256"]

rerun = reproducible_simulation(
    reloaded_input,
    seed=loaded["parameters"]["seed"],
    noise_sigma=loaded["parameters"]["noise_sigma"],
)

print("Recorded result:", loaded["result"]["value"])
print("Rerun result   :", rerun)
print("Same result    :", rerun == loaded["result"]["value"])

### Important distinction

Reproducing the **numerical result** and reproducing the **performance number** are related but not identical goals.

For performance, also record enough information about:

- resource allocation;
- hardware;
- software stack;
- measurement method;
- repeated runs / variability;
- system conditions where relevant.

That is why Session 13's measurement discipline and today's manifest belong together.

# 10 — ACTIVITY: make the Practice 3 manifest usable

In groups of three, inspect this deliberately incomplete record:

```text
Code:      optimise.py
Input:     test.npy
Command:   python optimise.py
Before:    2.80 s
After:     1.70 s
```

Roles:

- **Investigator:** identify missing evidence.
- **Recorder:** propose the minimum manifest fields.
- **Reviewer:** ask whether another student could rerun it without contacting the author.

### Success criterion

> Another group can reproduce the experiment without asking you a question.

In [ ]:
# A template students can reuse in Practice 3.
p3_template = {
    "run_id": "",
    "code_commit": "",
    "input_path": "",
    "input_checksum_or_version": "",
    "command": "",
    "parameters": {},
    "random_seed": None,
    "software_environment": {
        "compiler_or_python": "",
        "modules_or_packages": [],
        "loaded_modules": [],
        "module_list_file": "modules.txt",
    },
    "resources": {
        "nodes": None,
        "ranks": None,
        "threads": None,
        "gpus": None,
    },
    "measurement": {
        "repetitions": None,
        "reported_statistic": "median",
        "timings_seconds": [],
    },
    "outputs": [],
    "notes": "",
}

p3_template_path = WORK / "P3_run_manifest_template.json"
p3_template_path.write_text(json.dumps(p3_template, indent=2))
print("Created reusable template:", p3_template_path)
print(p3_template_path.read_text())

# 11 — MODEL: native environment → container → workflow

These layers solve different problems.

| Layer | Main question | Examples |
|---|---|---|
| **Native environment** | Which compiler/libraries/packages are active? | `module load`, module manifest, Spack |
| **Container** | Can I package a consistent user-space software stack? | Apptainer |
| **Workflow** | Can I describe and rerun a sequence of steps and dependencies? | Nextflow, Snakemake |
| **Scheduler / machine** | Where and with which resources does it execute? | Slurm, cluster/cloud |

### PREDICT

Does a container replace:

- Slurm? **No.**
- MPI/OpenMP? **No.**
- the need to record input data and parameters? **No.**

We return to containers and portability in depth later in the course. Today the important point is the separation of responsibilities.
### Important

For a native HPC run, a small environment record can be as simple as:

```bash
module -t list 2> modules.txt
gcc --version > compiler.txt
python --version > python.txt
```

A container captures more of the user-space software stack, but it **does not replace** the need to record inputs, parameters, resources and the execution command.


# 12 — Knowledge check

Answer before opening the solution.

1. Why is an optimized library often preferable to rewriting a standard numerical kernel?
2. What does the DFT/FFT experiment demonstrate beyond “libraries are fast”?
3. Why can 10,000 tiny files be more troublesome than one file containing the same number of bytes?
4. What is the conceptual difference between HDF5 and MPI-IO?
5. Why is a hotspot becoming faster capable of exposing an I/O bottleneck?
6. Name four fields that belong in a minimum run manifest.
7. What is the difference between `module avail`, `module show` and `module list`?
8. What does an environment module change, and why should `module list` be recorded?
9. What should you do after applying a performance optimization?

<details>
<summary><strong>Concise answers</strong></summary>

1. Mature libraries provide tested algorithms and heavily optimized implementations, reducing reinvention and often improving performance/portability.
2. It shows both **algorithmic complexity** and **implementation quality** matter; the FFT is not merely a faster implementation of the same O(N²) algorithm.
3. Each file adds namespace/metadata/open/close work, and many concurrent files can stress shared metadata services.
4. HDF5 is a higher-level structured scientific data model/API; MPI-IO is a lower-level parallel file-access interface. Parallel HDF5 can use MPI-IO underneath.
5. When compute time falls, previously smaller costs become a larger fraction of total runtime; the dominant bottleneck can move.
6. Examples: code commit, input/version/checksum, command/parameters, seed, software environment, resources, outputs and timings.
7. `module avail` lists software that can be loaded; `module show` reveals what one module changes; `module list` shows what is active now.
8. Environment modules change the active software/toolchain through environment variables; recording the loaded modules identifies an important part of the software environment.
9. **Remeasure under comparable conditions**, check correctness, and keep the change only if the evidence supports it.
</details>

# 13 — What you should leave with

- **Reuse mature software** for standard kernels.
- **Design the I/O**, not just the computation.
- File count, access pattern, coordination and storage path all matter.
- **HDF5 / NetCDF, MPI-IO and the filesystem are different layers.**
- **Modules select the native software environment**; record what was loaded.
- Reproducibility requires **code + input + parameters + environment + resources + output**.
- Containers and workflows help, but they do not replace good experimental records.

> **Fast code that cannot be rerun is incomplete HPC evidence.**

The workflow connecting Sessions 13 and 14 is:

> **MEASURE → LOCATE → EXPLAIN → CHANGE → REMEASURE → RECORD**

**Next:** Practice 3 — Profiling and optimization mini-project.